In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 68.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 153.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 164.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 176.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit]2m6/7 [qiskit]ne]
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=89d0ac36310870a4f86423a94b165823b830d8bd24940d4d14df2c59ba78a684
  Stored in directory: /hom

In [17]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

#My new imports (just to keep track)
from qiskit_aer import AerSimulator
import numpy as np

def measure_w(qc, qubit, classical_bit):
    qc.ry(-np.pi/4, qubit)
    qc.measure(qubit, classical_bit)

def measure_v(qc, qubit, classical_bit):
    qc.ry(np.pi/4, qubit)
    qc.measure(qubit, classical_bit)

def measure_z(qc, qubit, classic_bit):
    qc.measure(qubit, classic_bit)

def measure_x(qc, qubit, classic_bit):
    qc.h(qubit)
    qc.measure(qubit, classic_bit)
    

def create_bell_pair():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.x(1)
    qc.z(0)
    return qc

def bit_to_pm1(bit):
    return 1 if bit == '0' else -1

#Test Z basis
sim = AerSimulator()

qc = create_bell_pair()
measure_z(qc, 0, 0)
measure_z(qc, 1, 1)

qc = qc.reverse_bits()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)

#Test X basis
qc = create_bell_pair()
measure_x(qc, 0, 0)
measure_x(qc, 1, 1)

qc = qc.reverse_bits()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)

#Test W basis
qc = create_bell_pair()
measure_w(qc, 0, 0)
measure_w(qc, 1, 1)

qc = qc.reverse_bits()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)

#Test V basis
qc = create_bell_pair()
measure_v(qc, 0, 0)
measure_v(qc, 1, 1)

qc = qc.reverse_bits()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)

#Test Z/V
qc = create_bell_pair()
measure_z(qc, 0, 0)
measure_v(qc, 1, 1)

qc = qc.reverse_bits()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)


sim = AerSimulator()

def expectation_value(measure_A, measure_B, shots=2000):

    qc = create_bell_pair()

    measure_A(qc, 0, 0)
    measure_B(qc, 1, 1)

    qc = qc.reverse_bits()
    result = sim.run(qc, shots=shots).result()
    counts = result.get_counts()

    total = 0

    for bitstring, count in counts.items():
        a = bit_to_pm1(bitstring[0])
        b = bit_to_pm1(bitstring[1])
        total += a * b * count

    return total / shots

#Test needed terms
E_XW = expectation_value(measure_x, measure_w)
E_XV = expectation_value(measure_x, measure_v)
E_ZW = expectation_value(measure_z, measure_w)
E_ZV = expectation_value(measure_z, measure_v)

S = abs(E_XW - E_XV + E_ZW + E_ZV)

print("S =", S)


import random

def random_A():
    return random.choice([1,2,3])

def random_B():
    return random.choice([1,2,3])

A_map = {
    1: measure_x,
    2: measure_w,
    3: measure_z
}

B_map = {
    1: measure_w,
    2: measure_z,
    3: measure_v
}

def run_round():

    i = random_A()
    j = random_B()

    qc = create_bell_pair()

    A_map[i](qc, 0, 0)
    B_map[j](qc, 1, 1)

    qc = qc.reverse_bits()
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()

    bitstring = list(counts.keys())[0]

    return i, j, bitstring

#Run the simulation itself
N = 500
rounds = int(9*N/2)

results = [run_round() for _ in range(rounds)]

alice_key = []
bob_key = []

#Keep appropriate rounds
for i, j, bits in results:
    if (i == 2 and j == 1) or (i == 3 and j == 2):

        alice_bit = bits[0]
        bob_bit = bits[1]

        bob_bit = '0' if bob_bit == '1' else '1'  # Bob inverts

        alice_key.append(alice_bit)
        bob_key.append(bob_bit)

print("Keys match:", alice_key == bob_key)
print("Key length:", len(alice_key))


def compute_S_from_results(results):

    sums = {
        (1,1): 0,
        (1,3): 0,
        (3,1): 0,
        (3,3): 0
    }

    counts = {
        (1,1): 0,
        (1,3): 0,
        (3,1): 0,
        (3,3): 0
    }

    for i, j, bits in results:

        if (i, j) in sums:

            a = 1 if bits[0] == '0' else -1
            b = 1 if bits[1] == '0' else -1

            sums[(i,j)] += a * b
            counts[(i,j)] += 1

    E_XW = sums[(1,1)] / counts[(1,1)]
    E_XV = sums[(1,3)] / counts[(1,3)]
    E_ZW = sums[(3,1)] / counts[(3,1)]
    E_ZV = sums[(3,3)] / counts[(3,3)]

    S = abs(E_XW - E_XV + E_ZW + E_ZV)

    return S

S_protocol = compute_S_from_results(results)
print("S from protocol data =", S_protocol)

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

{'10': 467, '01': 533}
{'01': 493, '10': 507}
{'10': 496, '01': 504}
{'01': 511, '10': 489}
{'11': 64, '00': 74, '10': 446, '01': 416}
S = 2.8440000000000003
Keys match: True
Key length: 535
S from protocol data = 2.92046621446415
Total rounds: 2250
Key length: 535
